# D3 — GraphRAG Executor, Evaluation & Safety

This notebook demonstrates the complete D3 pipeline:

1. **Parameter Injection** — AutoML winning config loaded into hybrid search
2. **GraphRAG Pipeline** (Method A — Pre-Filter):
   - LLM generates Cypher → Neo4j subgraph → filtered search → LLM answer
3. **Provenance Filtering** — Safety check verifying citations against metadata
4. **Feedback Loop** — River online learning + ADWIN drift detection
5. **Evaluation & Ablation** — vector_only vs graph_only vs hybrid

**Prerequisites:**
- MongoDB, Qdrant, Neo4j running via `docker compose up -d mongodb qdrant neo4j`
- Data seeded via `python seed_data.py`
- `LLM_API_KEY` set in `.env` (or as environment variable)

## 0. Setup & Imports

In [ ]:
import sys, os

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

# Load .env if it exists
env_path = os.path.join(PROJECT_ROOT, ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, val = line.split("=", 1)
                os.environ.setdefault(key.strip(), val.strip())
    print("Loaded .env file.")

print(f"Working directory: {os.getcwd()}")
print(f"LLM_MODEL: {os.getenv('LLM_MODEL', 'gpt-4o-mini')}")
print(f"LLM_API_KEY set: {'Yes' if os.getenv('LLM_API_KEY') else 'NO — set this before running!'}")

---
## 1. Parameter Injection (Config to Search)

`hybrid_search.py` automatically loads the AutoML winning configuration from `configs/run_card.yaml` at import time. This means every search uses the optimized parameters without manual tuning.

In [ ]:
import yaml

# Show the raw config
with open("configs/run_card.yaml") as f:
    config = yaml.safe_load(f)

print("AutoML Winning Config:")
for key, val in config["winning_config"].items():
    print(f"  {key}: {val}")

print("\nAutoML Best Metrics vs Baseline:")
for name, metrics in config["metrics"].items():
    print(f"  {name}:")
    for mk, mv in metrics.items():
        print(f"    {mk}: {mv}")

In [ ]:
from hybrid_search import HybridSearcher, CONFIG_K, CONFIG_ALPHA, WINNING_CONFIG

print("Values loaded into hybrid_search.py:")
print(f"  CONFIG_K     = {CONFIG_K}  (default top_k for search)")
print(f"  CONFIG_ALPHA = {CONFIG_ALPHA}  (BM25 fusion weight)")
print(f"  Full config  = {WINNING_CONFIG}")

# Demonstrate that search() uses CONFIG_K by default
searcher = HybridSearcher()
results = searcher.search("transformer architecture")
print(f"\nsearch('transformer architecture') returned {len(results)} results (default k={CONFIG_K})")

---
## 2. GraphRAG Executor (Core Pipeline)

The `GraphRAGExecutor` implements Method A (Pre-Filter):

```
Query
  |
  +-> Step 1: LLM generates Cypher
  |            |
  |            v
  |        Step 2: Neo4j -> paper_ids
  |            |
  v            v
  Step 3: Filtered BM25 + Dense + RRF
  |
  v
  Step 4: LLM generates answer with citations
  |
  v
  Step 5: Provenance filter verifies citations
```

If the graph filter returns no results, it falls back to unfiltered search.

In [ ]:
from graphrag_executor import GraphRAGExecutor

executor = GraphRAGExecutor()
print("GraphRAGExecutor initialized.")
print(f"  LLM model: {executor._model}")
print(f"  Temperature: {executor._temperature}")

### 2.1 Step 1 — Cypher Generation

The LLM receives the full Neo4j schema and produces a Cypher query that returns `paper_id` values.

In [ ]:
test_query = "What papers discuss attention mechanisms in NLP?"

cypher, intent = executor._generate_cypher(test_query)
print(f"Query  : {test_query}")
print(f"Intent : {intent}")
print(f"Cypher : {cypher}")

### 2.2 Step 2 — Neo4j Subgraph Extraction

In [ ]:
if cypher:
    paper_ids = executor._execute_cypher(cypher)
    print(f"Neo4j returned {len(paper_ids)} paper(s):")
    for pid in paper_ids:
        print(f"  - {pid}")
else:
    paper_ids = []
    print("No Cypher generated (query had no graph-structural angle).")

### 2.3 Step 3 — Filtered Hybrid Search

In [ ]:
chunks, fallback = executor._filtered_search(
    test_query,
    paper_ids if paper_ids else None,
    top_k=5,
)

print(f"Retrieved {len(chunks)} chunks (fallback={fallback})")
for i, c in enumerate(chunks, 1):
    print(f"  [{i}] {c.citation()}")
    print(f"      {c.text[:100]}...\n")

### 2.4 Step 4 — LLM Answer Generation

The LLM is strictly instructed to cite every claim using `[Authors] "Title" (pp. X-Y)` format, using only the retrieved chunks.

In [ ]:
if chunks:
    raw_answer = executor._generate_answer(test_query, chunks)
    print("Raw LLM Answer:")
    print("=" * 72)
    print(raw_answer)
    print("=" * 72)
else:
    print("No chunks to generate answer from.")
    raw_answer = ""

---
## 3. Provenance Filtering (Safety)

The provenance filter extracts every citation from the LLM answer and cross-checks:
- Does the **paper title** match a retrieved chunk?
- Do the **page numbers** fall within the chunk's page range?

Unverified citations are replaced with `[CITATION REMOVED - unverified]`.

In [ ]:
from graphrag_executor import provenance_filter

if raw_answer and chunks:
    filtered_answer, verified, dropped, score = provenance_filter(raw_answer, chunks)

    print(f"Provenance Score: {score:.2%}")
    print(f"Verified citations: {len(verified)}")
    print(f"Dropped citations:  {len(dropped)}")

    if verified:
        print("\nVerified:")
        for c in verified:
            print(f"  + {c}")

    if dropped:
        print("\nDropped (failed provenance):")
        for c in dropped:
            print(f"  X {c}")

    print("\nFiltered answer:")
    print("=" * 72)
    print(filtered_answer)
    print("=" * 72)
else:
    print("Skipping provenance demo (no answer generated).")

### 3.1 Provenance filter — synthetic test

To show the filter catching a fabricated citation:

In [ ]:
from hybrid_search import SearchResult

# Create fake chunks with known metadata
fake_chunks = [
    SearchResult(
        chunk_id="c1", paper_id="p1",
        title="Attention Is All You Need",
        text="The dominant sequence transduction models...",
        score=0.9, page_start=1, page_end=3,
        authors=["Vaswani", "Shazeer"], source="rrf",
    ),
]

# Answer with one real and one fabricated citation
test_answer = (
    'Transformers use self-attention [Vaswani, Shazeer] "Attention Is All You Need" (pp. 1-3). '
    'They also use convolutions [Doe, Smith] "A Paper That Does Not Exist" (pp. 10-15).'
)

filtered, verified, dropped, score = provenance_filter(test_answer, fake_chunks)

print(f"Provenance score: {score:.0%}")
print(f"Verified: {verified}")
print(f"Dropped:  {dropped}")
print(f"\nFiltered output:\n{filtered}")

---
## 4. Full End-to-End Pipeline

The `executor.query()` method runs all 5 steps in sequence and returns a `GraphRAGResponse`.

In [ ]:
import time

query = "How does the attention mechanism work in transformers?"

t0 = time.perf_counter()
response = executor.query(query, top_k=5)
elapsed = (time.perf_counter() - t0) * 1000

print(f"Query: {query}")
print(f"Latency: {elapsed:.0f}ms")
print(f"\n{'='*72}")
print(f"Intent             : {response.intent}")
print(f"Cypher             : {response.cypher_generated}")
print(f"Graph papers found : {response.graph_papers_found}")
print(f"Filter applied     : {response.graph_filter_applied}")
print(f"Fallback used      : {response.fallback}")
print(f"Chunks used        : {response.chunks_used}")
print(f"Provenance score   : {response.provenance_score:.0%}")
print(f"Verified citations : {len(response.verified_citations)}")
print(f"Dropped citations  : {len(response.dropped_citations)}")
print(f"BM25 top score     : {response.bm25_top_score}")
print(f"Dense top score    : {response.dense_top_score}")
print(f"{'='*72}")
print(f"\nAnswer:\n{response.answer}")

---
## 5. Feedback Loop (River + ADWIN)

After each query, the user can submit y/n helpfulness feedback. This synchronously:
1. Updates the River LogisticRegression model weights
2. Triggers ADWIN to check for concept drift in the error stream

The adapter learns from `(bm25_score, dense_score, query_features)` → `helpful?`

In [ ]:
from online_learning import RiverHybridAdapter

adapter = RiverHybridAdapter()
print(f"River adapter initialized.")
print(f"  ADWIN delta: {adapter.adwin.delta}")
print(f"  Initial accuracy: {adapter.current_accuracy}")
print(f"  Total steps: {adapter.total_steps}")

In [ ]:
# Simulate a stream of feedback using the GraphRAG response scores
import random
random.seed(42)

simulated_feedback = [
    {"query": "attention mechanism", "bm25": 12.5, "dense": 0.87, "helpful": 1},
    {"query": "BERT pre-training", "bm25": 8.3, "dense": 0.91, "helpful": 1},
    {"query": "image classification CNN", "bm25": 3.1, "dense": 0.45, "helpful": 0},
    {"query": "transformer architecture", "bm25": 15.2, "dense": 0.93, "helpful": 1},
    {"query": "reinforcement learning policy", "bm25": 1.0, "dense": 0.32, "helpful": 0},
    {"query": "language model fine-tuning", "bm25": 10.7, "dense": 0.85, "helpful": 1},
    {"query": "knowledge graph embedding", "bm25": 6.2, "dense": 0.71, "helpful": 1},
    {"query": "retrieval augmented generation", "bm25": 14.1, "dense": 0.89, "helpful": 1},
    {"query": "robot manipulation", "bm25": 0.5, "dense": 0.21, "helpful": 0},
    {"query": "multi-head attention", "bm25": 13.8, "dense": 0.92, "helpful": 1},
]

print(f"{'Step':>4} {'Query':<35} {'Helpful':>7} {'Accuracy':>8} {'Drift':>5}")
print("-" * 65)

for fb in simulated_feedback:
    entry = adapter.learn(
        query_text=fb["query"],
        bm25_top_score=fb["bm25"],
        dense_top_score=fb["dense"],
        helpful=fb["helpful"],
    )
    drift_mark = "***" if entry.drift_detected else ""
    print(f"{entry.step:>4} {fb['query']:<35} {fb['helpful']:>7} {entry.accuracy:>8.3f} {drift_mark:>5}")

In [ ]:
summary = adapter.get_summary()
print("\nAdapter Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

# Predicted alpha for a new query
alpha = adapter.predict_alpha("attention mechanism", bm25_top_score=12.0, dense_top_score=0.88)
print(f"\nPredicted alpha for 'attention mechanism': {alpha:.4f}")
print(f"  (higher = favor BM25, lower = favor dense)")

---
## 6. Evaluation & Ablation Harness

The `evaluate.py` script tests the pipeline in three ablation modes:

| Mode | Description |
|------|-------------|
| `vector_only` | BM25 + Dense hybrid search, no graph filtering |
| `graph_only` | Only graph-filtered chunks, no fallback |
| `hybrid` | Full GraphRAG pipeline (graph filter with fallback) |

Metrics: **p95 latency**, **Faithfulness** (citation verification), **Answer-Relevance** (keyword overlap).

In [ ]:
from evaluate import load_queries

queries = load_queries("eval_queries.csv")
print(f"Loaded {len(queries)} evaluation queries:\n")
for i, q in enumerate(queries, 1):
    kw = q.get('expected_keywords', [])
    kw_str = ', '.join(kw) if kw else '(none)'
    print(f"  [{i}] {q['query']}")
    print(f"      keywords: {kw_str}")

### 6.1 Run a single mode (hybrid)

In [ ]:
from evaluate import evaluate_mode, print_summary_table

# Run just the hybrid mode on 2 queries (quick demo)
demo_queries = queries[:2]
results, agg = evaluate_mode(demo_queries, mode="hybrid", top_k=5)

print_summary_table([agg])

### 6.2 Full ablation (all three modes)

Compare `vector_only` vs `graph_only` vs `hybrid` side by side.

In [ ]:
# Run all three modes on the same queries
all_aggs = []
for mode in ["vector_only", "graph_only", "hybrid"]:
    _, agg = evaluate_mode(demo_queries, mode=mode, top_k=5)
    all_aggs.append(agg)

print_summary_table(all_aggs)

### 6.3 Faithfulness & Answer-Relevance scorers

In [ ]:
from evaluate import compute_faithfulness, compute_answer_relevance

# Faithfulness: what fraction of citations passed provenance
faith = compute_faithfulness("some answer", verified_count=3, total_citation_count=4)
print(f"Faithfulness (3/4 verified): {faith:.2%}")

faith_perfect = compute_faithfulness("some answer", verified_count=5, total_citation_count=5)
print(f"Faithfulness (5/5 verified): {faith_perfect:.2%}")

faith_none = compute_faithfulness("some answer", verified_count=0, total_citation_count=0)
print(f"Faithfulness (no citations): {faith_none:.2%}")

# Answer-Relevance: keyword overlap
rel = compute_answer_relevance(
    query="How does attention work?",
    answer="The attention mechanism computes scaled dot-product attention.",
    expected_keywords=["attention", "self-attention", "multi-head", "scaled dot-product"],
)
print(f"\nAnswer-Relevance (keyword): {rel:.2%}")

rel_query = compute_answer_relevance(
    query="How does attention work in transformers?",
    answer="The transformer model uses multi-head attention across all layers.",
)
print(f"Answer-Relevance (query token overlap): {rel_query:.2%}")

---
## 7. FastAPI Integration

All D3 components are exposed via two endpoints:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/graphrag` | POST | Full GraphRAG pipeline with provenance safety |
| `/feedback` | POST | Submit y/n feedback, triggers River + ADWIN |

Start the server: `uvicorn app:app --host 0.0.0.0 --port 8000 --reload`

In [ ]:
import requests

BASE = "http://localhost:8000"

try:
    # Test GraphRAG endpoint
    resp = requests.post(f"{BASE}/graphrag", json={
        "query": "What papers discuss attention mechanisms?",
        "top_k": 3,
    }, timeout=60).json()

    print("POST /graphrag response:")
    print(f"  Chunks used      : {resp['chunks_used']}")
    print(f"  Graph filter     : {resp['graph_filter_applied']}")
    print(f"  Provenance score : {resp['provenance_score']}")
    print(f"  Elapsed          : {resp['elapsed_ms']:.0f}ms")
    print(f"  Answer preview   : {resp['answer'][:200]}...")

    # Test feedback endpoint with scores from GraphRAG
    fb_resp = requests.post(f"{BASE}/feedback", json={
        "query": "What papers discuss attention mechanisms?",
        "helpful": 1,
        "bm25_top_score": resp["bm25_top_score"],
        "dense_top_score": resp["dense_top_score"],
    }, timeout=10).json()

    print(f"\nPOST /feedback response:")
    print(f"  Step             : {fb_resp['step']}")
    print(f"  Accuracy         : {fb_resp['current_accuracy']}")
    print(f"  Drift detected   : {fb_resp['drift_detected']}")

except requests.ConnectionError:
    print("API server not running. Start it with: uvicorn app:app --port 8000")

---
## Cleanup

In [ ]:
executor.close()
print("Connections closed.")
print("\nD3 notebook complete.")